In [600]:
from __future__ import annotations

In [601]:
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple, Protocol, Literal, Iterable
import pandas as pd

import numpy as np
import yaml
import time
import os
import json
import faiss

In [602]:
DEFAULT_PARAMS_PATH        = "parameters/parameters.yaml"
DEFAULT_SCORE_WEIGHTS_PATH = "parameters/retrieval_score_weights.yaml"

In [603]:
def _load_params_yaml(path:str=DEFAULT_PARAMS_PATH) -> Dict[str,Any]:
    '''load parameter from yaml file'''
    p = Path(path)
    with p.open("r",encoding="utf-8") as f:
        obj = yaml.safe_load(f) or {}
    return obj if isinstance(obj,dict) else {}

def _get_nested(d: Dict[str,Any], keys: Sequence[str], default:Any)->Any:
    '''Safe nested dict getter'''
    cur: Any = d
    for k in keys:
        if not isinstance(cur,dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def _coalesce(*vals:Any) -> Any:
    for v in vals:
        if v is not None:
            return v
    return None

@dataclass(frozen=True)
class UserBundlePaths:
    out_dir          : str
    bundle_json_path : str
    hyde_q_emb_path  : str

def resolve_user_bundle_paths(out_dir:str, student_id:str) -> UserBundlePaths:
    out_dir_abs = os.path.abspath(out_dir)
    bundle_json = os.path.join(out_dir_abs, f"{student_id}.json")
    hyde_q_emb  = os.path.join(out_dir_abs, f"{student_id}_hyde_q_emb.npy")
    return UserBundlePaths(
        out_dir          = out_dir_abs,
        bundle_json_path = bundle_json,
        hyde_q_emb_path  = hyde_q_emb
    )

def load_user_bundle(out_dir:str, student_id:str)-> Dict[str,Any]:
    '''Load a user's cached bundle JSON'''
    paths = resolve_user_bundle_paths(out_dir,student_id)
    print(f"paths -> {paths}")
    print(f"paths.bundle_json_path -> {paths.bundle_json_path}")
    if not os.path.exists(paths.bundle_json_path):
        raise FileNotFoundError(f"Missing bundle JSON: {paths.bundle_json_path}")
    with open(paths.bundle_json_path,"r", encoding="utf-8") as f:
        try:
            bundle = json.load(f)
        except Exception as e:
            raise ValueError(f"Invalid JSON in bundle: {paths.bundle_json_path} ({e})") from e
    if not isinstance(bundle, dict):
        raise ValueError(f"Bundle JSON must be an object/dict: {paths.bundle_json_path}")
    print(f"bundle -> {bundle}")
    # Light validation
    if str(bundle.get("student_id", "")) != str(student_id):
        raise ValueError(
            f"Bundle student_id mismatch: expected={student_id}, got={bundle.get('student_id')}"
        )
    return bundle
## bundle
# >>> {'bundle_version': 'v2_hyde_embedded_queries',
#  'student_id': 'stu_p001',
#  'generated_at': '2026-01-25T16:28:09+00:00',
#  'prompt_key': 'hyde_b',
#  'preferred_language': 'th',
#  'num_events': 9,
#  'user_context_json': {'student_id': 'stu_p001',
#   'preferred_language': 'th',
#   'current_status': 'student3yr',
#   'education': {'level': 'bachelor', 'major': 'วิทยาการคอมพิวเตอร์'},
#   'target_roles': [{'role_id': 'data_analyst',
#     'role_name': 'Data Analyst',
#     'priority': 1}],
#   'skills': [{'skill_id': 'python',
#     'skill_name': 'Python',
#     'proficiency': 'L2'},
#    {'skill_id': 'sql', 'skill_name': 'SQL', 'proficiency': 'L2'}],
#   'interests': ['ทำพอร์ต', 'ฝึกสัมภาษณ์'],
#   'onboard_grp': 'Job_Hunter',
#   'onboard_grp_description': 'เตรียมฝึกงานสายข้อมูล'},
#  'user_context_text': 'นักศึกษา : วิทยาการคอมพิวเตอร์ (student3yr)\nเป้าหมายอาชีพ : Data Analyst\nทักษะ : Python:L2;SQL:L2\nความสนใจ : ทำพอร์ต, ฝึกสัมภาษณ์\nกลุ่มผู้ใช้ : Job_Hunter (เตรียมฝึกงานสายข้อมูล)\nภาษา : ไทย',
#  'history_summary_text': 'สรุปพฤติกรรม 30 วันล่าสุด (รวม 9 เหตุการณ์):\nธีมที่มีการมีส่วนร่วมสูง:\n- data_career: คะแนน=12.7, เหตุการณ์=9, ไลก์=1, คลิก=1,dwell สูงสุด=52000ms\nฟีดล่าสุดที่มีปฏิสัมพันธ์ (ตัดสั้น):\n- (1) TH_F001 | แนวทางทำพอร์ต Data Analyst ด้วยโปรเจกต์ Python และ SQL | แนวทางทำพอร์ต Data Analyst ด้วยโปรเจกต์ Python และ SQL\n- (2) TH_F003 | เทคนิคเตรียมสัมภาษณ์ฝึกงาน Data Analyst (SQL + Python) | เทคนิคเตรียมสัมภาษณ์ฝึกงาน Data Analyst (SQL + Python)\n- (3) TH_F009 | โปรเจกต์ Python วิเคราะห์ข้อมูลจริง: ตั้งแต่ทำความสะอาดถึงสรุป insight | โปรเจกต์ Python วิเคราะห์ข้อมูลจริง: ตั้งแต่ทำความสะอาดถึงสรุป insight\n- (4) TH_F008 | รวมโจทย์ SQL ฝึกสัมภาษณ์ระดับฝึกงาน (พร้อมเฉลยแนวคิด) | รวมโจทย์ SQL ฝึกสัมภาษณ์ระดับฝึกงาน (พร้อมเฉลยแนวคิด)\n- (5) TH_F017 | Portfolio Project: วิเคราะห์ยอดขาย + สร้าง KPI Dashboard | Portfolio Project: วิเคราะห์ยอดขาย + สร้าง KPI Dashboard',
#  'hyde_output': {'output_language': 'th',
#   'hyde_queries': [{'query_id': 'Q1',
#     'query_text': 'แนวทางทำพอร์ต Data Analyst โปรเจกต์ Python SQL',
#     'weight': 1.0,
#     'intent_label': 'history_aligned'},
#    {'query_id': 'Q2',
#     'query_text': 'เทคนิคเตรียมสัมภาษณ์ฝึกงาน Data Analyst โจทย์ SQL Python',
#     'weight': 1.0,
#     'intent_label': 'history_aligned'},
#    {'query_id': 'Q3',
#     'query_text': 'ตัวอย่างโปรเจกต์ Data Analyst สำหรับฝึกงานพร้อมโค้ด',
#     'weight': 1.0,
#     'intent_label': 'practical'},
#    {'query_id': 'Q4',
#     'query_text': 'สร้าง Dashboard ด้วย Power BI หรือ Tableau สำหรับ Data Analyst',
#     'weight': 0.6,
#     'intent_label': 'exploratory'},
#    {'query_id': 'Q5',
#     'query_text': 'เส้นทางอาชีพ Data Analyst ทักษะที่จำเป็นในอนาคต',
#     'weight': 0.6,
#     'intent_label': 'exploratory'}]},
#  'hyde_query_embeddings': {'path': 'stu_p001_hyde_q_emb.npy',
#   'model': 'gemini-embedding-001',
#   'dim': 768,
#   'dtype': 'float32',
#   'num_queries': 5,
#   'normalized': True}}

# =============================================================================
# Policy: cached .npy must exist (unless declared num_queries == 0)
# =============================================================================
def _assert_cached_npy_exists(user_bundle_dir:str, student_id: str, bundle: Dict[str,Any]) -> None:
    emb_meta = bundle.get("hyde_query_embeddings") or {}
    num_q    = emb_meta.get("num_queries")
    try:
        num_q_i = int(num_q) if num_q is not None else None
    except Exception:
        num_q_i = None
    path_rel = emb_meta.get("path")

    # No path: only allowed when bundle declares "no queries".
    if not isinstance(path_rel, str) or not path_rel.strip():
        if num_q_i is None or num_q_i > 0:
            raise FileNotFoundError(
                f"Missing hyde_query_embeddings.path in bundle for student_id={student_id}. "
                f"Expected cached .npy under {user_bundle_dir}."
            )
        return
    
    npy_path = Path(user_bundle_dir)/path_rel
    # Declared empty => allow missing file.
    if num_q_i is not None and num_q_i == 0:
        return
    
    if not npy_path.exists():
        raise FileNotFoundError(
            f"Cached HyDE .npy not found for student_id={student_id}. "
            f"Expected: {path_rel} (abs: {npy_path.resolve()}). "
            f"Run pipeline_1_user_hyde.py or pipeline_2_user_hyde_refresh.py to generate."
        )
    
def _resolve_embedding_path(out_dir: str, emb_path_value: Optional[str], student_id: str) -> str:
    """
    Resolve embedding path from bundle metadata.
    - If emb_path_value is absolute and exists -> use it
    - If emb_path_value is relative -> join with out_dir
    - Else fall back to {student_id}_hyde_q_emb.npy in out_dir
    """
    out_dir_abs = os.path.abspath(out_dir)

    if emb_path_value:
        p = str(emb_path_value)
        # relative filename only
        if not os.path.isabs(p):
            candidate = os.path.join(out_dir_abs, p)
            return candidate
        return p

    # fallback
    return os.path.join(out_dir_abs, f"{student_id}_hyde_q_emb.npy")

def load_user_hyde_query_embeddings(
        out_dir: str,
        student_id: str,
        bundle: Dict[str,Any],
        *,
        dim_expected: Optional[int] = None
) -> np.ndarray:
    meta = bundle.get("hyde_query_embeddings") or {}
    if not isinstance(meta, dict):
        meta = {}
    meta_dim = meta.get("dim",None)    
    meta_nq  = meta.get("num_queries",None)

    # prefer explicit dim_expected, else meta.dim, else infer later
    dim_target: int = 0
    if dim_expected is not None:
        dim_target = int(dim_expected)
    elif meta_dim is not None:
        dim_target = int(meta_dim)                 # 768

    emb_path_value = meta.get("path",None)         # stu_p001_hyde_q_emb.npy
    emb_path       = _resolve_embedding_path(out_dir, emb_path_value, student_id)

    if not os.path.exists(emb_path):
        raise FileNotFoundError(f"Missing HyDE query embeddings .npy: {emb_path}")
    
    ### --------- Load npy --------- ###
    arr = np.load(emb_path)

    if not isinstance(arr, np.ndarray):
        arr = np.asarray(arr)

    if arr.dtype != np.float32:
        arr = arr.astype(np.float32)

    if arr.ndim != 2:
        raise ValueError(f"HyDE embedding matrix must be 2D: got shape={arr.shape} path={emb_path}")

    nq, dim = int(arr.shape[0]), int(arr.shape[1])

    # Empty query case: enforce (0, dim_target) if dim_target known and differs
    if nq == 0 and dim_target and dim != dim_target:
        arr = np.zeros((0, dim_target), dtype=np.float32)
        return arr

    # If dim_target is known, enforce it
    if dim_target and dim != dim_target:
        raise ValueError(f"HyDE embedding dim mismatch: got {dim}, expected {dim_target} path={emb_path}")

    # Optional sanity check against metadata num_queries
    if meta_nq is not None:
        try:
            meta_nq_int = int(meta_nq)
            if meta_nq_int != nq:
                # keep this as strict error; better to catch corrupted bundle early
                raise ValueError(
                    f"HyDE embedding num_queries mismatch: file has {nq}, meta says {meta_nq_int} path={emb_path}"
                )
        except Exception:
            # ignore if meta num_queries is not parseable
            pass

    return arr

### online_score_aggregation.py

In [604]:
#======================================================================
# (A) HyDE query-weight extraction
#======================================================================
def extract_query_weights_and_labels(
        bundle:Dict[str,Any]
    ) -> Tuple[np.ndarray,List[str],List[str]]:
    # Prefer current schema
    hq = None
    hyde_out = bundle.get("hyde_output")
    if isinstance(hyde_out, dict):
        hq = hyde_out.get("hyde_queries")
    if not isinstance(hq,list):
        hq = bundle.get("hyde_queries")
    if not isinstance(hq, list) or not hq:
        return np.array([], dtype=np.float32), [], []
    
    weights : List[float] = []
    qids    : List[str]   = []
    intents : List[str]   = []

    for i,q in enumerate(hq):
        if not isinstance(q,dict):
            qids.append(f"Q{i+q}")
            intents.append("unknows")
            weights.append("1.0")
            continue
        qid = q.get("query_id")
        if not isinstance(qid,str) or not qid.strip():
            qid = f"Q{i+1}"
        qids.append(qid)

        intent = q.get("intent_label")
        if not isinstance(intent,str) or not intent.strip():
            intent = "unknows"
        intents.append(intent)

        try:
            wf = float(q.get("weight",1.0))
        except Exception:
            wf = 1.0
        weights.append(wf)
    return np.asarray(weights, dtype=np.float32), qids, intents
# (array([1. , 1. , 1. , 0.6, 0.6], dtype=float32),
#  ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
#  ['history_aligned',
#   'history_aligned',
#   'practical',
#   'exploratory',
#   'exploratory'])



### index_store.py

In [605]:
def _coerce_int(x:Any, default:int = 0) -> int:
    """
    Best-effort case to int.
    """
    try:
        if x is None:
            return int(default)
        return int(float(x))
    except Exception:
        return int(default)
    
def _ensure_popularity(row:Dict[str,Any]) -> Dict[str,Any]:
    """
    Ensure output meta has a canonical integer `popularity` field.
    """
    out = dict(row)
    if "popularity" in out:
        out["popularity"] = _coerce_int(out.get("popularity"), default=0)
        return out
    if "views" in out:
        out["popularity"] = _coerce_int(out.get("views"), default=0)
        return out
    out["popularity"] = 0
    return out

@dataclass
class FeedIndexStore:
    index_dir    : str
    _faiss_index : faiss.Index
    _meta        : List[Dict[str,Any]]
    _manifest    : Dict[str,Any]
    @classmethod
    def load(cls, feed_index_dir:str) -> "FeedIndexStore":
        faiss_path    = os.path.join(feed_index_dir, "index.faiss")       # <- artifacts/feed_index\index.faiss
        meta_path     = os.path.join(feed_index_dir, "feeds_meta.jsonl")  # <- artifacts/feed_index\feeds_meta.jsonl
        manifest_path = os.path.join(feed_index_dir, "manifest.json")     # <- artifacts/feed_index\manifest.json
        
        # Validate required artifacts early for clearer errors.
        if not os.path.exists(faiss_path):
            raise FileNotFoundError(f"faiss index not found: {faiss_path}")
        if not os.path.exists(meta_path):
            raise FileNotFoundError(f"meta not found: {meta_path}")
        
        ### --------- 1) Load FAISS index from disk --------- ###
        index = faiss.read_index(faiss_path)
        print(f"index -> {index}")
        ### --------- 2) Load metadata JSONL aligned with FAISS internal ids. --------- ###
        meta: List[Dict[str,Any]] = []
        with open(meta_path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    meta.append(_ensure_popularity(json.loads(line)))
                except Exception as e:
                    raise ValueError(f"Invalid JSONL at line {line_no}: {e}") from e
        print(f"meta  -> {meta}")
        # Ensure meta length matches index size.
        # If this fails, internal-id -> meta alignment is broken.
        try:
            ntotal = int(getattr(index,"ntotal",0))
        except Exception:
            ntotal = 0
        if ntotal > 0 and len(meta) != ntotal:
            raise ValueError(
                f"Meta length mismatch: len(feeds_ eta)={len(meta)} vs faiss.ntotal={ntotal}."
                "feeds_meta.jsonl row order must align with FAISS internal ids."
            )
        ### --------- 3) Load manifest (optional). Manifest is not required for retrieval --------- ###
        manifest: Dict[str,Any] = {}
        if os.path.exists(manifest_path):
            try:
                with open(manifest_path,"r",encoding="utf-8") as f:
                    manifest = json.load(f) or {}
            except Exception:
                manifest = {}
        return cls(
            index_dir = feed_index_dir,
            _faiss_index = index,
            _meta = meta,
            _manifest = manifest
        )
    
    def search(self,query_embeddings:np.ndarray, top_k:int) -> Tuple[np.ndarray, np.ndarray]:
        """Search nearest neighbors via vector index (POC: FAISS)"""
        if not isinstance(query_embeddings,np.ndarray):
            query_embeddings = np.asarray(query_embeddings)
        if query_embeddings.ndim != 2:
            raise ValueError("query_embeddings must be a 2D array of shape (nq,dim)")
        k = int(top_k)
        if k <= 0:
            raise ValueError("top_k must be > 0")
        # FAISS expects float32 contiguous arrays.
        q = np.asarray(query_embeddings, dtype = np.float32)
        q = np.ascontiguousarray(q)

        # Execute FAISS search: returns (scores, indices)
        scores, indices = self._faiss_index.search(q,k)
        return np.asarray(scores, dtype=np.float32), np.asarray(indices, dtype=np.int64)
    
    def get_feed_id(self, internal_idx:int) -> Optional[str]:
        """Convert a vector index internal id into a feed_id using aligned metadata"""
        i = int(internal_idx)
        if i < 0 or i >= len(self._meta):
            return None
        row = self._meta[i]
        fid = row.get("feed_id")
        return str(fid) if fid is not None else None

    
_INDEX_CACHE: Dict[str,FeedIndexStore] = {}
def get_index_store_cached(feed_index_dir:str) -> FeedIndexStore:
    key = str(Path(feed_index_dir).resolve())
    cached = _INDEX_CACHE.get(key)
    if cached is not None:
        return cached
    store = FeedIndexStore.load(feed_index_dir)
    _INDEX_CACHE[key] = store
    return store

### Retrieval.py

In [606]:
class FeedIndexStoreLike(Protocol):
    """
    Minimal interface required for retrieval.
    """
    def search(self, query_embeddings:np.ndarray,top_k:int) -> Tuple[np.ndarray, np.ndarray]:
        ...
    def get_feed_id(self, internal_idx: int) -> str | None:
        ...

AggMode = Literal["WEIGHTED_MAX","WEIGHTED_MEAN"]           

@dataclass(frozen=True)
class RetrievalDebug:
    """Explainability information for a single retrieval candidate"""
    best_qi : int
    best_raw_score : float
    best_weight : float
    aggregated_score : float

@dataclass(frozen=True)
class RetrievalDebug:
    best_qi: int
    best_raw_score: float
    best_weight: float
    aggregated_score: float

def _validate_query_weights(nq:int, query_weights: Optional[np.ndarray]) -> np.ndarray:
    """Validate and sanitize query weights"""
    if query_weights is None:
        return np.ones((nq,), dtype=np.float32)

    w = np.asarray(query_weights, dtype=np.float32).reshape(-1)
    if w.shape[0] != nq:
        raise ValueError(f"query_weights must have length {nq}, got {w.shape[0]}")

    # Deterministic sanitization
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.maximum(w, 0.0)

    return w.astype(np.float32)


def l2_normalize_rows(x: np.ndarray) -> np.ndarray:
    """Row-wise L2 normalization (safe for zero vectors)."""
    if not isinstance(x, np.ndarray):
        x = np.asarray(x)

    if x.ndim != 2:
        raise ValueError("l2_normalize_rows expects a 2D array of shape (n, d)")

    x = x.astype(np.float32, copy=False)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0

    return (x / norms).astype(np.float32, copy=False)

#----------------------------------------------------------------------
# Main retrieval + aggregation logic
#----------------------------------------------------------------------
def retrieve_by_hyde_queries_weighted(
    *,
    index_store: FeedIndexStoreLike,
    hyde_query_embeddings: np.ndarray,
    query_weights: Optional[np.ndarray],
    top_k: int,
    agg_mode: AggMode = "WEIGHTED_MAX",
    return_debug: bool = False,
) -> Tuple[List[Tuple[str, float]], Optional[Dict[str, RetrievalDebug]]]:
    """Retrieve candidate feeds using HyDE query embeddings with weighted aggregation"""
    if not isinstance(hyde_query_embeddings, np.ndarray):
        raise ValueError("hyde_query_embeddings must be a numpy array")
    if hyde_query_embeddings.ndim != 2:
        raise ValueError("hyde_query_embeddings must be a 2D array")
    if int(top_k) <= 0:
        raise ValueError("top_k must be > 0")
    nq = int(hyde_query_embeddings.shape[0])
    if nq == 0:
        return ([], {} if return_debug else None)
    
    # Validate / sanitize weights
    w = _validate_query_weights(nq, query_weights)

    # Normalize query embeddings (idempotent if already normalized)
    q = l2_normalize_rows(hyde_query_embeddings.astype(np.float32, copy=False))

    # ----------------------------------------------------------------------
    # Vector search (backent-agnostic)
    # ----------------------------------------------------------------------
    scores, indices = index_store.search(q, top_k=int(top_k))
    print(f"scores  -> {scores}")
    print(f"indices -> {indices}")

    # ----------------------------------------------------------------------
    # Aggregation state
    # ----------------------------------------------------------------------
    agg_scores   : Dict[str, float] = {}
    best_qi_map  : Dict[str, int]   = {}
    best_raw_map : Dict[str, float] = {}
    best_w_map   : Dict[str, float] = {}

    # For WEIGHTED_MEAN
    sum_w_map    : Dict[str, float] = {}
    sum_ws_map   : Dict[str, float] = {}

    # ----------------------------------------------------------------------
    # Aggregate per-query results (AGG mode)
    # ----------------------------------------------------------------------
    for qi in range(indices.shape[0]):      # how many d of d,n (defualt = 5)
        wi = float(w[qi])
        print(f"qi:{qi} -> wi:{wi}")
        for ki in range(indices.shape[1]):
            print(f"- ki : {ki}")
            idx = int(indices[qi,ki])
            raw = float(scores[qi,ki])
            print(f"- idx : {idx} -> raw : {raw}")

            feed_id = index_store.get_feed_id(idx)
            print(f"- feed_id -> {feed_id}")
            if feed_id is None:
                continue

            if agg_mode == "WEIGHTED_MAX":
                cand = wi * raw
                prev = agg_scores.get(feed_id)
                print(f"  - cand -> {cand}")
                print(f"  - cand -> {prev}")
                if prev is None or cand > prev:
                    agg_scores[feed_id]   = cand
                    best_qi_map[feed_id]  = qi
                    best_raw_map[feed_id] = raw
                    best_w_map[feed_id]   = wi
            
            elif agg_mode == "WEIGHTED_MEAN":
                if wi <= 0.0:
                    continue
                sum_ws_map[feed_id] = sum_ws_map.get(feed_id,0.0) + wi * raw
                sum_w_map[feed_id]  = sum_w_map.get(feed_id,0.0)  + wi
                
                # Track best raw score for explainability
                prev_best = best_raw_map.get(feed_id)
                if prev_best is None or raw > prev_best:
                    best_qi_map[feed_id]  = qi
                    best_raw_map[feed_id] = raw
                    best_w_map[feed_id]   = wi
            else:
                raise Value(f"Unknow agg_mode: {agg_mode}")
        print("-"*50)
    if agg_mode == "WEIGHTED_MEAN":
        for fid, ws in sum_ws_map.items():
            sw = sum_w_map.get(fid, 0.0)
            if sw > 0.0:
                agg_scores[fid] = float(ws/sw)
    print(f"agg_scores   -> {agg_scores}")
    print(f"best_qi_map  -> {best_qi_map}")
    print(f"best_raw_map -> {best_raw_map}")
    print(f"best_w_map   -> {best_w_map}")
    print(f"sum_w_map    -> {sum_w_map}")
    print(f"sum_ws_map   -> {sum_ws_map}")
    # Finalize WEIGHTED_MEAN scores
    # if agg_mode == "WEIGHTED_MEAN":
    #     for fid, ws 
    # ----------------------------------------------------------------------
    # Deterministic sorting
    # ----------------------------------------------------------------------
    # Tie-breakers:
    # 1) aggregated score (desc)
    # 2) best query index (asc)
    # 3) feed_id (lex asc)
    def _sort_key(item: Tuple[str,float]) -> Tuple[float, int, str]:
        fid, sc = item
        qi = best_qi_map.get(fid,10**9)
        return (-float(sc), int(qi), str(fid))
    candidates = sorted(agg_scores.items(), key=_sort_key)
    print(f"candidates -> {candidates}")
    # ----------------------------------------------------------------------
    # Optional debug output
    # ----------------------------------------------------------------------
    debug : Optional[Dict[str, RetrievalDebug]] = None
    if return_debug:
        debug = {
            fid: RetrievalDebug(
                best_qi = int(best_qi_map.get(fid,-1)),
                best_raw_score = float(best_raw_map.get(fid, 0.0)),
                best_weight = float(best_w_map.get(fid, 0.0)),
                aggregated_score=float(sc),
            )
            for fid, sc in candidates
        }
    return candidates, debug


In [607]:
# # index_store           = index_store,
# # hyde_query_embeddings = hyde_q_emb,
# # query_weights         = weights,
# # top_k                 = int(top_k_per_query),
# # agg_mode              = mode,
# # return_debug          = True

# candidates, debug = retrieve_by_hyde_queries_weighted(
#     index_store           = index_store,
#     hyde_query_embeddings = hyde_q_emb,
#     query_weights         = weights,
#     # top_k                 = int(top_k_per_query),
#     top_k                 = int(5),
#     agg_mode              = mode,
#     return_debug          = True
# )


In [608]:
# candidates
# >>> [('TH_F001', 0.9919580221176147),
#  ('TH_F003', 0.9699496626853943),
#  ('TH_F023', 0.9138078093528748),
#  ('TH_F008', 0.9090964198112488),
#  ('TH_F006', 0.9064895510673523),
#  ('TH_F009', 0.9038510918617249),
#  ('TH_F012', 0.9035521745681763),
#  ('TH_F005', 0.8786376714706421),
#  ('TH_F017', 0.5257789343905017),
#  ('TH_F010', 0.522958609380666)]

In [609]:
# debug
# >>> {'TH_F001': RetrievalDebug(best_qi=0, best_raw_score=0.9919580221176147, best_weight=1.0, aggregated_score=0.9919580221176147),
#  'TH_F003': RetrievalDebug(best_qi=1, best_raw_score=0.9699496626853943, best_weight=1.0, aggregated_score=0.9699496626853943),
#  'TH_F023': RetrievalDebug(best_qi=2, best_raw_score=0.9138078093528748, best_weight=1.0, aggregated_score=0.9138078093528748),
#  'TH_F008': RetrievalDebug(best_qi=1, best_raw_score=0.9090964198112488, best_weight=1.0, aggregated_score=0.9090964198112488),
#  'TH_F006': RetrievalDebug(best_qi=2, best_raw_score=0.9064895510673523, best_weight=1.0, aggregated_score=0.9064895510673523),
#  'TH_F009': RetrievalDebug(best_qi=2, best_raw_score=0.9038510918617249, best_weight=1.0, aggregated_score=0.9038510918617249),
#  'TH_F012': RetrievalDebug(best_qi=2, best_raw_score=0.9035521745681763, best_weight=1.0, aggregated_score=0.9035521745681763),
#  'TH_F005': RetrievalDebug(best_qi=0, best_raw_score=0.8786376714706421, best_weight=1.0, aggregated_score=0.8786376714706421),
#  'TH_F017': RetrievalDebug(best_qi=3, best_raw_score=0.876298189163208, best_weight=0.6000000238418579, aggregated_score=0.5257789343905017),
#  'TH_F010': RetrievalDebug(best_qi=4, best_raw_score=0.8715976476669312, best_weight=0.6000000238418579, aggregated_score=0.522958609380666)}

### interactions_store.py

In [610]:
def _read_interactions_table(p:Path) -> pd.DataFrame:
    """Robust reader for interactions file that may be TSV, CSV"""
    df = pd.read_csv(p, sep=None, engine="python", encoding="utf-8-sig")
    def _strip_cols(d:pd.DataFrame) -> pd.DataFrame:
        d.columns = [str(c).strip() for c in d.columns]
        return d
    df = _strip_cols(df)
    if len(df.columns) == 1:
        # common case: TSV but sniff failed
        df2 = pd.read_csv(p, sep="\t", engine="python", encoding="utf-8-sig")
        df2 = _strip_cols(df2)
        if len(df2.columns) > 1:
            return df2
        # fallback: comma
        df3 = pd.read_csv(p, sep=",", engine="python", encoding="utf-8-sig")
        df3 = _strip_cols(df3)
        return df3
    return df

def load_user_interactions(interactions_path: str, *, student_id:str) -> pd.DataFrame:
    """Load interactions file and return rows for this student_id"""
    p = Path(interactions_path)
    if not p.exists():
        raise FileNotFoundError(f"interactions_path not found: {p}")
    df = _read_interactions_table(p)
    # normalize column names (strip already handled now lower for safety)
    df.columns = [c.lower().strip() for c in df.columns]
    id_col: Optional[str] = None
    if "student_id" in df.columns:
        id_col = "student_id"
    elif "user_id" in df.columns:
        id_col = "user_id"

    if id_col is None:
        raise ValueError("interactions.csv must contain 'student_id' or 'user_id' column")

    sid = str(student_id).strip()
    out = df[df[id_col].astype(str).str.strip() == sid].copy()
    return out


In [611]:
def _read_interactions_table(p:Path) -> pd.DataFrame:
    """Robust reader for interactions file that may be TSV, CSV"""
    df = pd.read_csv(p, sep=None, engine="python", encoding="utf-8-sig")
    def _strip_cols(d:pd.DataFrame) -> pd.DataFrame:
        d.columns = [str(c).strip() for c in d.columns]
        return d
    df = _strip_cols(df)
    if len(df.columns) == 1:
        # common case: TSV but sniff failed
        df2 = pd.read_csv(p, sep="\t", engine="python", encoding="utf-8-sig")
        df2 = _strip_cols(df2)
        if len(df2.columns) > 1:
            return df2
        # fallback: comma
        df3 = pd.read_csv(p, sep=",", engine="python", encoding="utf-8-sig")
        df3 = _strip_cols(df3)
        return df3
    return df

def load_user_interactions(interactions_path: str, *, student_id:str) -> pd.DataFrame:
    """Load interactions file and return rows for this student_id"""
    p = Path(interactions_path)
    if not p.exists():
        raise FileNotFoundError(f"interactions_path not found: {p}")
    df = _read_interactions_table(p)
    # normalize column names (strip already handled now lower for safety)
    df.columns = [c.lower().strip() for c in df.columns]
    id_col: Optional[str] = None
    if "student_id" in df.columns:
        id_col = "student_id"
    elif "user_id" in df.columns:
        id_col = "user_id"

    if id_col is None:
        raise ValueError("interactions.csv must contain 'student_id' or 'user_id' column")

    sid = str(student_id).strip()
    out = df[df[id_col].astype(str).str.strip() == sid].copy()
    return out


# ----------------------------------------------------------------------
# Public API: seen-feed extraction for exclude-seen policy
# ----------------------------------------------------------------------
def extract_seen_feed_ids(
    user_events: pd.DataFrame,
    *,
    event_types: Optional[Iterable[str]] = None,
    now_utc: Optional[datetime] = None,
    window_days: Optional[int] = None,
    max_unique: Optional[int] = None,
) -> Set[str]:
    """Extract unique feed_ids the user already interacted with.
    This is used in pipeline_3_online_retrieval for exclude-seen filtering.
    """
    if user_events is None or len(user_events) == 0:
        return set()
    if "feed_id" not in user_events.columns:
        return set()

    df = user_events.copy()
    df["feed_id"] = df["feed_id"].astype(str).str.strip()
    df = df[df["feed_id"].astype(bool)]

    # Filter event types if possible.
    if event_types is not None and "event_type" in df.columns:
        allow = {str(x).lower().strip() for x in event_types if str(x).strip()}
        df["event_type"] = df["event_type"].astype(str).str.lower().str.strip()
        df = df[df["event_type"].isin(allow)]

    ts_col = "ts" if "ts" in df.columns else None

    # Optional windowing by time (only if ts exists).
    if window_days is not None and int(window_days) > 0 and ts_col is not None:
        now_utc = now_utc or datetime.now(timezone.utc)
        ts = pd.to_datetime(df[ts_col], errors="coerce", utc=True)
        df = df.assign(_ts=ts).dropna(subset=["_ts"])
        if not df.empty:
            cutoff = now_utc - pd.Timedelta(days=int(window_days))
            df = df[df["_ts"] >= cutoff]

    # Optional deterministic cap by most-recent unique feeds.
    if max_unique is not None and int(max_unique) > 0 and ts_col is not None:
        ts = pd.to_datetime(df[ts_col], errors="coerce", utc=True)
        df = df.assign(_ts=ts).dropna(subset=["_ts"]).sort_values("_ts", ascending=False)

        seen: Set[str] = set()
        for fid in df["feed_id"].tolist():
            if fid in seen:
                continue
            seen.add(fid)
            if len(seen) >= int(max_unique):
                break
        return seen

    return set(df["feed_id"].tolist())


def _get_seen_feed_ids_from_params(
        *,
        params     : Dict[str,Any],
        student_id : str,
        now_utc    : datetime
) -> Tuple[bool, Set[str], Dict[str,Any]]:
    """Resolve "exclude seen" behavior from parameters and return seen feed ids."""
    cfg = _get_nested(params, ["retrieval","exclude_seen"],{}) or {}
    enabled = bool(cfg.get("enabled",False))
    meta = {
        "exclude_seen_enabled": enabled,
        "exclude_seen_count"  : 0
    }
    if not enabled:
        return False, set(), meta
    interactions_path = str(cfg.get("interactions_path","data/interactions.csv"))
    event_types = cfg.get("event_types",["view","click","like","share","comment"])
    window_days = cfg.get("window_days",30)
    max_unique  = cfg.get("max_unique",5000)
    print(f"interactions_path -> {interactions_path}")
    print(f"event_types       -> {event_types}")
    print(f"window_days       -> {window_days}")
    print(f"max_unique        -> {max_unique}")
    # Load interactions filtered to this student.
    df = load_user_interactions(interactions_path, student_id = student_id)
    # convert events into "seen" feed_ids with a time windown
    seen = extract_seen_feed_ids(
        df,
        event_types=event_types,
        now_utc=now_utc,
        window_days=int(window_days) if window_days is not None else None,
        max_unique=int(max_unique) if max_unique is not None else None,
    )
    print(f"seen -> {seen}")
    meta.update(
        {
            "exclude_seen_enabled":True,
            "exclude_seen_count"  :int(len(seen)),
            "exclude_seen_interactions_path" : interactions_path,
            "exclude_seen_event_types":[str(x) for x in (event_types or [])],
            "exclude_seen_window_days": int(window_days) if window_days is not None else None,
            "exclude_seen_max_unique" : int(max_unique) if max_unique is not None else None,
        }
    )
    return True, seen, meta


### feeds_meta_store.py

In [612]:
_FEED_META_CACHE : Dict[str,Dict[str,Dict[str,Any]]] = {}
def load_feeds_meta_map(feed_index_dir:str) -> Dict[str,Dict[str,Any]]:
    cache_key = str(Path(feed_index_dir).resolve())
    cached    = _FEED_META_CACHE.get(cache_key)
    if cached is not None:
        return cached
    meta_path = Path(feed_index_dir) / "feeds_meta.jsonl"
    print(f"meta_path -> {meta_path}")
    out : Dict[str, Dict[str,Any]] = {}
    # If file doesn't exist, return empty map (callers should handle missing meta).
    if meta_path.exists():
        with meta_path.open("r",encoding="utf-8") as f:
            for line_no,line in enumerate(f, start=1):
                # print(f"line_no -> {line_no}, line -> {line}")
                line = line.strip()
                if not line:
                    continue
                # Best-effort parse JSON; skip invalid lines rather than failing serving
                try:
                    obj = json.loads(line)
                except Exception:
                    continue
                # Ensure the parsed JSON is a dict-like object.
                if not isinstance(obj, dict):
                    continue
                # Support both "feed_id" and legacy "id" fields
                fid = obj.get("feed_id") or obj.get("id")
                # Only accept non-empty string ids to keep map stable.
                if isinstance(fid,str) and fid.strip():
                    out[fid.strip()] = obj
    _FEED_META_CACHE[cache_key] = out
    return out


### subscores.py

In [613]:
# ----------------------------------------------------------------------
# User language extraction helper
# ----------------------------------------------------------------------
def get_user_lang(bundle:Dict[str,Any]) -> Optional[str]:
    """Infer user language from a cached user bundle"""
    v = bundle.get("output_language") or bundle.get("preferred_language")
    if isinstance(v, str) and v.strip():
        return v.strip().lower()

    uc = bundle.get("user_context") or bundle.get("UserContext")
    if isinstance(uc, dict):
        v2 = uc.get("preferred_language") or uc.get("language") or uc.get("lang")
        if isinstance(v2, str) and v2.strip():
            return v2.strip().lower()
    return None

# ----------------------------------------------------------------------
# Language match score
# ----------------------------------------------------------------------
def score_language_match(
        feed_meta: Dict[str,Any],
        *,
        user_lang: Optional[str]
) -> float:
    """Score exact language match between user and feed."""
    if not user_lang:
        return 0.0
    feed_lang = feed_meta.get("lang") or feed_meta.get("language")
    if isinstance(feed_lang,str) and feed_lang.strip().lower() == user_lang.strip().lower():
        return 1.0
    return 0.0

# ----------------------------------------------------------------------
# User language extraction help
# ----------------------------------------------------------------------
def get_user_lang(bundle: Dict[str,Any]) -> Optional[str]:
    """Infer user language from a cached user bundle."""
    v = bundle.get("output_language") or bundle.get("preferred_language")
    if isinstance(v, str) and v.strip():
        return v.strip().lower()

    uc = bundle.get("user_context") or bundle.get("UserContext")
    if isinstance(uc, dict):
        v2 = uc.get("preferred_language") or uc.get("language") or uc.get("lang")
        if isinstance(v2, str) and v2.strip():
            return v2.strip().lower()

    return None

# --------------------------------------------------------------------------------------
# Timestamp parsing
# --------------------------------------------------------------------------------------
def parse_ts_any(v: Any) -> Optional[datetime]:
    """
    Parse a timestamp from multiple common formats into UTC datetime.
    """
    if v is None:
        return None
    # UNIX timestamp (seconds)
    if isinstance(v, (int, float)):
        try:
            return datetime.fromtimestamp(float(v), tz=timezone.utc)
        except Exception:
            return None
    # ISO-8601-like string
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return None
        try:
            # Normalize trailing Z
            if s.endswith("Z"):
                s = s[:-1] + "+00:00"

            dt = datetime.fromisoformat(s)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)

            return dt.astimezone(timezone.utc)
        except Exception:
            return None
    return None

# ----------------------------------------------------------------------
# Recency score
# ----------------------------------------------------------------------
def score_recency(
        feed_meta: Dict[str, Any],
        *,
        now_utc: datetime,
        half_life_days: float = 30.0
) -> float:
    """Compute an exponential time-decay recency score in [0,1]
    score = 2 ^ (-age_days / half_life_days)
    """
    for key in ("published_at", "created_at", "timestamp", "date","ts"):
        dt = parse_ts_any(feed_meta.get(key))
        if dt is None:
            continue
        age_days = (now_utc - dt).total_seconds() / 86400.0
        if age_days < 0:
            age_days = 0.0
        if half_life_days <= 0:
            return 0.0
        return float(2.0**(-(age_days/half_life_days)))
    return 0.0

# ----------------------------------------------------------------------
# Popularity score
# ----------------------------------------------------------------------
def score_popularity(feed_meta:Dict[str,Any]) -> float:
    """Compute a bounded popularity score in [0,1] using log scaling
    score = log1p(v) / log1p(1_000_000)
    """
    for key in ("popularity","views","likes","clicks","impressions"):
        v = feed_meta.get(key)
        if isinstance(v, (int,float)) and v>0:
            return float(
                min(
                    1.0,
                    np.log1p(float(v)) / np.log1p(1_000_000.0),
                )
            )
    return 0.0


### online_score_aggregation.py

In [614]:
def _to_float(x: Any, default: float) -> float:
    try:
        return float(x)
    except Exception:
        return float(default)
    
def _safe_yaml_load(path:str) -> Dict[str,Any]:
    """Best-effort YAML loader"""
    try:
        p = Path(path)
        if not p.exists():
            return {}
        data = yaml.safe_load(p.read_text(encoding="utf-8")) or {}
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}

def _parse_score_aggregation(d: Dict[str, Any]) -> ScoreAggregationConfig:
    sa = d.get("score_aggregation") if isinstance(d, dict) else None
    if not isinstance(sa, dict):
        return ScoreAggregationConfig(enabled=False, weights={})

    enabled = bool(sa.get("enabled", False))
    mode = str(sa.get("mode", "linear") or "linear").lower().strip()

    w_raw = sa.get("weights", {}) or {}
    weights: Dict[str, float] = {}
    if isinstance(w_raw, dict):
        for k, v in w_raw.items():
            if isinstance(k, str) and k.strip():
                weights[k.strip()] = _to_float(v, 0.0)

    clamp_inputs = bool(sa.get("clamp_inputs", True))
    renormalize = bool(sa.get("renormalize", True))
    missing = _to_float(sa.get("missing_subscore_value", 0.0), 0.0)

    tb = sa.get("tie_breakers", ["vector_score", "recency"])
    if isinstance(tb, (list, tuple)):
        tie_breakers = tuple(str(x) for x in tb if str(x).strip())
    else:
        tie_breakers = ("vector_score", "recency")

    return ScoreAggregationConfig(
        enabled=enabled,
        mode=mode,
        weights=weights,
        clamp_inputs=clamp_inputs,
        renormalize=renormalize,
        missing_subscore_value=missing,
        tie_breakers=tie_breakers or ("vector_score", "recency"),
    )

@dataclass(frozen=True)
class ScoreAggregationConfig:
    enabled: bool = False
    mode : str = "linear"
    weights : Dict[str,float] = None
    clamp_inputs: bool = True
    renormalize:bool = True
    missing_subscore_value:float = 0.0
    tie_breakers: Tuple[str,...] = ("vector_score","recency")

def load_score_aggregation_config(path:str) -> Dict[str,Any]:
    """Load score aggregation YAML and return a dict-shaped config"""
    raw = _safe_yaml_load(path)
    print(f"raw -> {raw}")
    cfg = _parse_score_aggregation(raw)
    print(f"load_score_aggregation_config cfg -> {cfg}")
    return {
        "score_aggregation": {
            "enabled": bool(cfg.enabled),
            "mode": str(cfg.mode),
            "weights": dict(cfg.weights or {}),
            "clamp_inputs": bool(cfg.clamp_inputs),
            "renormalize": bool(cfg.renormalize),
            "missing_subscore_value": float(cfg.missing_subscore_value),
            "tie_breakers": list(cfg.tie_breakers),
        }
    }

def _get_feature_value(candidate: Dict[str, Any], key: str, missing: float) -> float:
    """
    Fetch a feature value from a candidate.
    """
    if not isinstance(candidate, dict):
        return float(missing)
    if key in candidate:
        return _to_float(candidate.get(key), missing)
    subs = candidate.get("subscores")
    if isinstance(subs, dict) and key in subs:
        return _to_float(subs.get(key), missing)
    return float(missing)

def _clamp01(x: float) -> float:
    if x < 0.0:
        return 0.0
    if x > 1.0:
        return 1.0
    return x

def aggregate_candidates(candidates: List[Dict[str, Any]], cfg: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Aggregate retrieval similarity and subscores into a final ranking scores"""
    sa = cfg.get("score_aggregation") if isinstance(cfg, dict) else None
    if not isinstance(sa,dict) or not bool(sa.get("enabled",False)):
        return candidates
    mode = str(sa.get("mode","linear") or "linear").lower().strip()
    if mode != "linear":
        return candidates
    weights_raw = sa.get("weights",{}) or {}
    weights : Dict[str,float] = {}
    if isinstance(weights_raw, dict):
        for k,v in weights_raw.items():
            if isinstance(k,str) and k.strip():
                weights[k.strip()] = _to_float(v,0.0)
    print(f"def aggregate_candidates weights -> {weights}")
    clamp_inputs = bool(sa.get("clamp_inputs",True))
    renormalize  = bool(sa.get("renormalize",True))
    missing      = _to_float(sa.get("missing_subscore_value",0.0),0.0)

    tie_breakers = sa.get("tie_breakers",["vector_score","recency"])
    if not isinstance(tie_breakers, (list,tuple)):
        tie_breakers = ["vector_score","recency"]
    tie_breakers = [str(x) for x in tie_breakers if str(x).strip()]
    if renormalize:
        s = float(sum(max(0.0,float(v)) for v in weights.values()))
        if s > 0.0:
            weights = {k:float(v) / s for k,v in weights.items()}
    print(f"def aggregate_candidates weights -> {weights}")
    
    out : List[Dict[str,Any]] = []
    for c in candidates:
        print(f"c -> {c}")
        row = dict(c) if isinstance(c,dict) else {"feed_id":None}

        final_score = 0.0
        for feat, w in weights.items():
            print(f"- feat -> {feat} | w -> {w}")
            if float(w) == 0.0:
                continue
            v = _get_featrue_value(c, feat, missing)
            print(f"- v -> {v}")
            if clamp_inputs:
                v = _clamp01(float(v))
            final_score += float(w) * float(v)
        row["final_score"] = float(final_score)
        out.append(row)
    
    # Deterministic sorting
    def _sort_key(c:Dict[str,Any]):
        keys : List[float] = [float(c.get("final_score",0.0))]
        for tb in tie_breakers:
            v = _get_feature_value(c,tb,missing)
            if clamp_inputs:
                v = _clamp01(float(v))
        return tuple(keys)

    out.sort(key=_sort_key, reverse=True)
    return out



### Pipeline_3_online_retrieval.py

In [615]:
def _extract_feed_header(fmeta: Dict[str,Any], header_keys: List[str]) -> Optional[str]:
    """Extract a short, human-readable header/title from feed metadata"""
    if not isinstance(fmeta,dict):
        return None
    for k in header_keys:
        v = fmeta.get(k)
        if isinstance(v,str) and v.strip():
            return v.strip()
    return None

<hr>

### Main

In [616]:
params = _load_params_yaml(DEFAULT_PARAMS_PATH)
params

{'app': {'name': 'poc_hyde_feed_recommendation', 'timezone': 'UTC'},
 'data': {'feeds_path': 'data/feeds.jsonl',
  'students_path': 'data/students.csv',
  'interactions_path': 'data/interactions.csv'},
 'artifacts': {'feed_index_dir': 'artifacts/feed_index',
  'user_query_bundles_dir': 'artifacts/user_query_bundles'},
 'llm': {'model_name': 'gemini-2.5-flash',
  'max_output_tokens': 2048,
  'temperature': 0.2},
 'hyde': {'history_threshold': 5,
  'recent_k': 5,
  'feed_text_max_chars': 240,
  'retry_limit': 2,
  'prompt_version': 'v2'},
 'retrieval': {'top_k_per_query': 30,
  'max_candidates': 200,
  'agg_mode': 'WEIGHTED_MAX',
  'recency_half_life_days': 30.0,
  'include_feed_header': True,
  'print_feed_header': True,
  'feed_header_keys': ['title', 'header', 'name', 'feed_title'],
  'exclude_seen': {'enabled': True,
   'interactions_path': 'data/interactions.csv',
   'event_types': ['view', 'click', 'like', 'share', 'comment'],
   'window_days': 30,
   'max_unique': 5000}},
 'scorin

In [617]:
# YAML toggles (CLI can overide)
yaml_include_header = bool(_get_nested(params, ["retrieval", "include_feed_header"], False))
yaml_print_header   = bool(_get_nested(params, ["retrieval", "print_feed_header"], False))

In [618]:
# If user wants to print headers, we must include them in rows
# Jupyter have to parser system
include_header_final = True
print_header_final   = True

In [619]:
# student_id = "stu_p001"
# res = run_online_retrieval(
#         student_id=args.student_id,
#         feed_index_dir=args.feed_index_dir,
#         user_bundle_dir=args.user_bundle_dir,
#         top_k_per_query=args.top_k_per_query,
#         max_candidates=args.max_candidates,
#         dim_expected=args.dim_expected,
#         agg_mode=args.agg_mode,
#         recency_half_life_days=args.recency_half_life_days,
#         include_feed_header=_coalesce(include_header_final, yaml_include_header),
#         score_weights_path=args.score_weights_path,
#         params_path=args.params_path,
#     )

In [620]:
@dataclass
class OnlineRetrievalResult:
    student_id : str
    candidates : List[Dict[str,Any]]
    meta       : Dict[str,Any]
    
student_id:str                         = "stu_p001"
feed_index_dir:Optional[str]           = None
user_bundle_dir:Optional[str]          = None
top_k_per_query:Optional[int]          = None
max_candidates:Optional[int]           = None
dim_expected:Optional[int]             = None
agg_mode:Optional[str]                 = None
now_utc:Optional[datetime]             = None
recency_half_life_days:Optional[float] = None
include_feed_header:Optional[bool]     = None
score_weights_path:Optional[str]       = None
params_path:str                        = DEFAULT_PARAMS_PATH

t0_total  = time.perf_counter()
timing_ms:Dict[str, float] = {}

def _ms(dt: float) -> float:
    """Convert perf_counter delta seconds -> milliseconds."""
    return float(dt*1000.0)

#----------------------------------------------------------------------
# Load params (best-effort)
#----------------------------------------------------------------------
t0 = time.perf_counter()
params = _load_params_yaml(params_path)
timing_ms["load_params_ms"] = _ms(time.perf_counter() - t0)

### --------- Directories: allow passing explicit dirs, else YAML defaults, else fallbacks. ---------###
feed_index_dir = str(_coalesce(feed_index_dir,_get_nested(
    d       = params,
    keys    = ["artifacts","feed_index_dir"],
    default = "artifacts/feed_index"
)))
# feed_index_dir   -> 'artifacts/feed_index'
user_bundle_dir = str(_coalesce(
    user_bundle_dir,
    _get_nested(
        d       = params,
        keys    = ["artifacts","user_query_bundles_dir"],
        default = "artifacts/user_query_bundles"
)))
# user_bundle_dir    -> 'artifacts/user_query_bundles'

### --------- Retrieval defaults --------- ###
top_k_per_query = int(_coalesce(_get_nested(
    d       = params,
    keys    = ["retrieval","top_k_per_query"],
    default = 50
)))
# top_k_per_query -> 30
max_candidates = int(_coalesce(_get_nested(
    d       = params,
    keys    = ["retrieval","max_candidates"],
    default = 200
)))
# max_candidates -> 200
agg_mode = str(_coalesce(agg_mode, _get_nested(params, ["retrieval", "agg_mode"], "WEIGHTED_MAX")))
# agg_mode    -> 'WEIGHTED_MAX'
recency_half_life_days = float(_coalesce(recency_half_life_days, _get_nested(params, ["retrieval", "recency_half_life_days"], 30.0)))
# recency_half_life_days    -> 30.0

### --------- Score weights path (best-effort; actual load is conditional on file existence) --------- ###
score_weights_path = str(_coalesce(score_weights_path,_get_nested(params, ["retrieval", "score_weights_path"], DEFAULT_SCORE_WEIGHTS_PATH),))
# score_weights_path    -> 'parameters/retrieval_score_weights.yaml'

### --------- Deterministic "now" for scoring + windowing --------- ###
now_utc = now_utc or datetime.now(timezone.utc)

### --------- Feed header config --------- ###
include_feed_header = bool(_coalesce(include_feed_header, _get_nested(params, ["retrieval", "include_feed_header"], False)))
# include_feed_header    -> True
header_keys = _get_nested(params, ["retrieval", "feed_header_keys"], ["title", "header", "name", "feed_title"])
if not isinstance(header_keys, list) or not header_keys:
    header_keys = ["title", "header", "name", "feed_title"]
header_keys = [str(x) for x in header_keys]
# header_keys    -> ['title', 'header', 'name', 'feed_title']

@dataclass
class OnlineRetrievalResult:
    student_id : str
    candidates : List[Dict[str,Any]]
    meta       : Dict[str,Any]
    
student_id:str                         = "stu_p001"
feed_index_dir:Optional[str]           = None
user_bundle_dir:Optional[str]          = None
top_k_per_query:Optional[int]          = None
max_candidates:Optional[int]           = None
dim_expected:Optional[int]             = None
agg_mode:Optional[str]                 = None
now_utc:Optional[datetime]             = None
recency_half_life_days:Optional[float] = None
include_feed_header:Optional[bool]     = None
score_weights_path:Optional[str]       = None
params_path:str                        = DEFAULT_PARAMS_PATH

t0_total  = time.perf_counter()
timing_ms:Dict[str, float] = {}

def _ms(dt: float) -> float:
    """Convert perf_counter delta seconds -> milliseconds."""
    return float(dt*1000.0)

#----------------------------------------------------------------------
# Load params (best-effort)
#----------------------------------------------------------------------
t0 = time.perf_counter()
params = _load_params_yaml(params_path)
timing_ms["load_params_ms"] = _ms(time.perf_counter() - t0)

### --------- Directories: allow passing explicit dirs, else YAML defaults, else fallbacks. ---------###
feed_index_dir = str(_coalesce(feed_index_dir,_get_nested(
    d       = params,
    keys    = ["artifacts","feed_index_dir"],
    default = "artifacts/feed_index"
)))
# feed_index_dir   -> 'artifacts/feed_index'
user_bundle_dir = str(_coalesce(
    user_bundle_dir,
    _get_nested(
        d       = params,
        keys    = ["artifacts","user_query_bundles_dir"],
        default = "artifacts/user_query_bundles"
)))
# user_bundle_dir    -> 'artifacts/user_query_bundles'

### --------- Retrieval defaults --------- ###
top_k_per_query = int(_coalesce(_get_nested(
    d       = params,
    keys    = ["retrieval","top_k_per_query"],
    default = 50
)))
# top_k_per_query -> 30
max_candidates = int(_coalesce(_get_nested(
    d       = params,
    keys    = ["retrieval","max_candidates"],
    default = 200
)))
# max_candidates -> 200
agg_mode = str(_coalesce(agg_mode, _get_nested(params, ["retrieval", "agg_mode"], "WEIGHTED_MAX")))
# agg_mode    -> 'WEIGHTED_MAX'
recency_half_life_days = float(_coalesce(recency_half_life_days, _get_nested(params, ["retrieval", "recency_half_life_days"], 30.0)))
# recency_half_life_days    -> 30.0

### --------- Score weights path (best-effort; actual load is conditional on file existence) --------- ###
score_weights_path = str(_coalesce(score_weights_path,_get_nested(params, ["retrieval", "score_weights_path"], DEFAULT_SCORE_WEIGHTS_PATH),))
# score_weights_path    -> 'parameters/retrieval_score_weights.yaml'

### --------- Deterministic "now" for scoring + windowing --------- ###
now_utc = now_utc or datetime.now(timezone.utc)

### --------- Feed header config --------- ###
include_feed_header = bool(_coalesce(include_feed_header, _get_nested(params, ["retrieval", "include_feed_header"], False)))
# include_feed_header    -> True
header_keys = _get_nested(params, ["retrieval", "feed_header_keys"], ["title", "header", "name", "feed_title"])
if not isinstance(header_keys, list) or not header_keys:
    header_keys = ["title", "header", "name", "feed_title"]
header_keys = [str(x) for x in header_keys]
# header_keys    -> ['title', 'header', 'name', 'feed_title']

#----------------------------------------------------------------------
# Load user bundle + envorce cached embedding policy
#----------------------------------------------------------------------
t0 = time.perf_counter()
bundle = load_user_bundle(user_bundle_dir, student_id)
timing_ms["load_bundle_ms"] = _ms(time.perf_counter() - t0)

t0 = time.perf_counter()
_assert_cached_npy_exists(user_bundle_dir, student_id, bundle)
timing_ms["assert_cached_npy_ms"] = _ms(time.perf_counter() - t0)

#----------------------------------------------------------------------
# Load cached HyDE embeddings (NO embedding calls allowed here)
#----------------------------------------------------------------------
t0 = time.perf_counter()
hyde_q_emb = load_user_hyde_query_embeddings(
    user_bundle_dir,
    student_id,
    bundle,
    dim_expected=dim_expected
)
timing_ms["load_hyde_emb_ms"] = _ms(time.perf_counter() - t0)

#----------------------------------------------------------------------
# Query weights/labels aligned with embedding rows
#----------------------------------------------------------------------
t0 = time.perf_counter()
weights, qids, intents = extract_query_weights_and_labels(bundle)
timing_ms["extract_query_weights_ms"] = _ms(time.perf_counter() - t0)

# If we have embeddings but no weights were provided, default to uniform weights.
if weights.size == 0 and hyde_q_emb.shape[0] > 0:
    weights = np.ones((hyde_q_emb.shape[0],),dtype=np.float32)
    qids    = [f"Q{i+1}" for i in range(hyde_q_emb.shape[0])]
    intents = ["unknown" for _ in range(hyde_q_emb.shape[0])]
# Hard alignment guard: prevents silent scoring bugs.
if hyde_q_emb.shape[0] != int(weights.shape[0]):
    raise ValueError(
        f"HyDE embeddings row count != weights count for student_id={student_id}: "
        f"emb_nq={hyde_q_emb.shape[0]} weights_nq={weights.shape[0]}"
    )
# ------------------------------------------------------------------
# Index caching (process-local)
# ------------------------------------------------------------------
t0 = time.perf_counter()
index_store = get_index_store_cached(feed_index_dir)
timing_ms["get_index_store_ms"] = _ms(time.perf_counter() - t0)

# ----------------------------------------------------------------------
# Vector retrieval (multi-query aggregation)
# ----------------------------------------------------------------------
t0 = time.perf_counter()
mode = "WEIGHTED_MEAN" if str(agg_mode).upper() == "WEIGHTED_MEAN" else "WEIGHTED_MAX"

scored, debug_map = retrieve_by_hyde_queries_weighted(
    index_store           = index_store,
    hyde_query_embeddings = hyde_q_emb,
    query_weights         = weights,
    top_k                 = int(top_k_per_query),
    agg_mode              = mode,
    return_debug          = True
)
timing_ms["vector_retrieval_ms"] = _ms(time.perf_counter() - t0)

# ----------------------------------------------------------------------
# Exclude seen feeds BEFORE max_candidates cap
# ----------------------------------------------------------------------
t0 = time.perf_counter()
exclude_enabled, seen_set, seen_meta = _get_seen_feed_ids_from_params(
        params=params,
        student_id=student_id,
        now_utc=now_utc,
    )
print(f"exclude_enabled -> {exclude_enabled}")
print(f"seen_set        -> {seen_set}")
print(f"seen_meta       -> {seen_meta}")

pre_filter_len = len(scored)
if exclude_enabled and seen_set:
    scored = [(fid,sc) for (fid,sc) in scored if fid not in seen_set]
post_filter_len = len(scored)

print(f"pre_filter_len  -> {pre_filter_len}")
print(f"post_filter_len -> {post_filter_len}")

# Cap after filtering to preserve "return up to N unseen candidates
scored = scored[: int(max(0,max_candidates))]
timing_ms["exclude_seen_ms"] = _ms(time.perf_counter() -t0)

# ----------------------------------------------------------------------
# Deterministic subscore
# ----------------------------------------------------------------------
t0 = time.perf_counter()
feeds_meta_map = load_feeds_meta_map(feed_index_dir)
timing_ms["load_feeds_meta_ms"] = _ms(time.perf_counter() - t0)
print(f"feeds_meta_map -> {feeds_meta_map}")

user_lang = get_user_lang(bundle)
print(f"user_lang -> {user_lang}")

paths -> UserBundlePaths(out_dir='c:\\Users\\TunKedsaro\\Desktop\\poc_hyde_feed_recommendation\\sandbox\\artifacts\\user_query_bundles', bundle_json_path='c:\\Users\\TunKedsaro\\Desktop\\poc_hyde_feed_recommendation\\sandbox\\artifacts\\user_query_bundles\\stu_p001.json', hyde_q_emb_path='c:\\Users\\TunKedsaro\\Desktop\\poc_hyde_feed_recommendation\\sandbox\\artifacts\\user_query_bundles\\stu_p001_hyde_q_emb.npy')
paths.bundle_json_path -> c:\Users\TunKedsaro\Desktop\poc_hyde_feed_recommendation\sandbox\artifacts\user_query_bundles\stu_p001.json
bundle -> {'bundle_version': 'v2_hyde_embedded_queries', 'student_id': 'stu_p001', 'generated_at': '2026-01-25T16:28:09+00:00', 'prompt_key': 'hyde_b', 'preferred_language': 'th', 'num_events': 9, 'user_context_json': {'student_id': 'stu_p001', 'preferred_language': 'th', 'current_status': 'student3yr', 'education': {'level': 'bachelor', 'major': 'วิทยาการคอมพิวเตอร์'}, 'target_roles': [{'role_id': 'data_analyst', 'role_name': 'Data Analyst', '

In [621]:
t0 = time.perf_counter()
candidates: List[Dict[str,Any]] = []
for fid, score in scored:
    print(f"fid -> {fid}")
    print(f"score -> {score}")
    # Feed matadata lookup by feed_id (independent of vector backend).
    fmeta = feeds_meta_map.get(fid, {}) if isinstance(feeds_meta_map,dict) else {}
    # Subscore are deterministic functions of (feed_meta, user_lang, now_utc)
    subscores = {
        "language_match": score_language_match(fmeta,user_lang=user_lang),
        "recency"       : score_recency(fmeta, now_utc=now_utc, half_life_days = float(recency_half_life_days)),
        "popularity"    : score_popularity(fmeta)
    }
    print(f"- subscore -> {subscores}")
    # Debug : indicate which HyDE query contributed most (if available).
    dgb_obj = None
    if debug_map is not None:
        dbg = debug_map.get(fid)
        if dbg is not None:
            qi = int(dbg.best_qi)
            dbg_obj = {
                "best_query_id" : qids[qi] if 0 <= qi < len(qids) else None,
                "best_query_intent" : intents[qi] if 0 <= qi < len(intents) else None,
                "best_raw_score" : float(dbg.best_raw_score),
                "best_weight": float(dbg.best_weight),
            }
    row : Dict[str,Any] = {
        "feed_id" : fid,
        "vector_score" : float(score),
        "subscores" : subscores
    }
    print(f"- row -> {row}")

    if include_feed_header:
        header = _extract_feed_header(fmeta if isinstance(fmeta,dict) else {}, header_keys)
        row["feed_header"] = header
    if dbg_obj is not None:
        row["debug"] = dbg_obj
    candidates.append(row)
    print()
timing_ms["build_candidates_ms"] = _ms(time.perf_counter() - t0)


fid -> TH_F001
score -> 0.9919580221176147
- subscore -> {'language_match': 1.0, 'recency': 8.568584468944074e-05, 'popularity': 0.8182601703827922}
- row -> {'feed_id': 'TH_F001', 'vector_score': 0.9919580221176147, 'subscores': {'language_match': 1.0, 'recency': 8.568584468944074e-05, 'popularity': 0.8182601703827922}}

fid -> TH_F003
score -> 0.9699496626853943
- subscore -> {'language_match': 1.0, 'recency': 8.173768225851107e-05, 'popularity': 0.8099573688230748}
- row -> {'feed_id': 'TH_F003', 'vector_score': 0.9699496626853943, 'subscores': {'language_match': 1.0, 'recency': 8.173768225851107e-05, 'popularity': 0.8099573688230748}}

fid -> TH_F023
score -> 0.9138078093528748
- subscore -> {'language_match': 1.0, 'recency': 0.1493748333645631, 'popularity': 0.7422222182490312}
- row -> {'feed_id': 'TH_F023', 'vector_score': 0.9138078093528748, 'subscores': {'language_match': 1.0, 'recency': 0.1493748333645631, 'popularity': 0.7422222182490312}}

fid -> TH_F008
score -> 0.90909641

In [622]:
# ----------------------------------------------------------------------
# Optional : final score aggregation (vector + subscore)
# Only attempt if the weights YAML exists and is enabled.
# ----------------------------------------------------------------------
t0 = time.perf_counter()
score_cfg: Dict[str, Any] = {}
score_enabled = False


try:
    if score_weights_path and Path(score_weights_path).exists():
        score_cfg = load_score_aggregation_config(score_weights_path)
        print(f"main score_cfg -> {score_cfg}")
        sa = score_cfg.get("score_aggregation") if isinstance(score_cfg, dict) else {}
        print(f"main sa -> {sa}")
        sa = sa if isinstance(sa, dict) else {}
        score_enabled = bool(sa.get("enabled", False))
        print(f"fscore_enabled -> {score_enabled}")
        print("Option1")
except Exception:
    score_cfg = {}
    score_enabled = False
    print("Option2")

# Downstream consumers should rely on this key when displaying or evaluation ordering.
ranking_score_key = "vector_score"

print(f"candidates -> {candidates}")
print(f"candidates -> {len(candidates)}")
if score_enabled:
    print(f"score_enabled -> {score_enabled}")
    print(f"score_cfg -> {score_cfg}")
    # aggregate_candidates:
    # - compute final_score per candidate
    # - sorts candidates deterministically using tie breaker
    candidates = aggregate_candidates(candidates, score_cfg)
    ranking_score_key = "final_score"
    for c in candidates:
        if "final_agg_score" in c:
            c.pop("final_agg_score",None)
timing_ms["score_aggregation_ms"] = _ms(time.perf_counter() - t0)

    # Total wall time
timing_ms["total_ms"] = _ms(time.perf_counter() - t0_total)


raw -> {'score_aggregation': {'enabled': True, 'mode': 'linear', 'weights': {'vector_score': 0.6, 'language_match': 0.3, 'recency': 0.05, 'popularity': 0.05}, 'clamp_inputs': True, 'renormalize': True, 'missing_subscore_value': 0.0, 'tie_breakers': ['vector_score', 'recency']}}
load_score_aggregation_config cfg -> ScoreAggregationConfig(enabled=True, mode='linear', weights={'vector_score': 0.6, 'language_match': 0.3, 'recency': 0.05, 'popularity': 0.05}, clamp_inputs=True, renormalize=True, missing_subscore_value=0.0, tie_breakers=('vector_score', 'recency'))
main score_cfg -> {'score_aggregation': {'enabled': True, 'mode': 'linear', 'weights': {'vector_score': 0.6, 'language_match': 0.3, 'recency': 0.05, 'popularity': 0.05}, 'clamp_inputs': True, 'renormalize': True, 'missing_subscore_value': 0.0, 'tie_breakers': ['vector_score', 'recency']}}
main sa -> {'enabled': True, 'mode': 'linear', 'weights': {'vector_score': 0.6, 'language_match': 0.3, 'recency': 0.05, 'popularity': 0.05}, 'cl

In [ ]:
# ----------------------------------------------------------------------
# Meta payload (for debugging + gold validation
# ----------------------------------------------------------------------
sa_out:Dict[str,Any] = {}
if isinstance(score_cfg,dict):
    sa_raw = score_cfg.get("score_aggregation")
    if isinstance(sa_raw,dict):
        sa_out = sa_raw
meta = {
    # retrieval config
    "top_k_per_query"       : int(top_k_per_query),
    "max_candidates"        : int(max_candidates),
    "num_queries"           : int(hyde_q_emb.shape[0]),
    "dim"                   : int(hyde_q_emb.shape[1]) if hyde_q_emb.ndim == 2 else 0,
    "bundle_path"           : f"{student_id}.json",
    "agg_mode"              : mode,
    # policy / safety
    "missing_npy_policy"    : "HARD_FAIL",
    "index_cached"          : True,
    # scorring context   
    "user_lang"             : user_lang,
    "recency_half_life_day" : float(recency_half_life_days),
    # debug-friedly header config
    "include_feed_header"   : bool(include_feed_header),
    "feed_header_keys"      : header_keys,
    # provenance
    "params_path"           : params_path,
    "feed_index_dir"        : feed_index_dir,
    "user_bundle_dir"       : user_bundle_dir,
    # score aggregation meta (safe subset)
    "score_weights_path"    : score_weights_path,
        "score_aggregation": {
            "enabled": bool(sa_out.get("enabled", False)) if sa_out else False,
            "mode": str(sa_out.get("mode", "linear")) if sa_out else "linear",
            "weights": dict(sa_out.get("weights", {}) or {})
            if (sa_out and isinstance(sa_out.get("weights"), dict))
            else {},
            "clamp_inputs": bool(sa_out.get("clamp_inputs", True)) if sa_out else True,
            "renormalize": bool(sa_out.get("renormalize", True)) if sa_out else True,
            "missing_subscore_value": float(sa_out.get("missing_subscore_value", 0.0)) if sa_out else 0.0,
            "tie_breakers": list(sa_out.get("tie_breakers", []) or []) if sa_out else [],
            "applied": bool(score_enabled),
        },
    # seen-feed meta from exclusion step
    **seen_meta,
    # timing
    "timing_ms": timing_ms,
    "timing_meta": {
        "retrieved_pairs_pre_filter": int(pre_filter_len),
        "retrieved_pairs_post_filter": int(post_filter_len),
        "returned_candidates": int(len(candidates)),
        "exclude_seen_enabled": bool(exclude_enabled),
        "exclude_seen_removed": int(max(0, pre_filter_len - post_filter_len)),
    },
    # tell consumers what score key represents ordering
    "ranking_score_key": ranking_score_key,
}


{'top_k_per_query': 30,
 'max_candidates': 200,
 'num_queries': 5,
 'dim': 768,
 'bundle_path': 'stu_p001.json',
 'agg_mode': 'WEIGHTED_MAX',
 'missing_npy_policy': 'HARD_FAIL',
 'index_cached': True,
 'user_lang': 'th',
 'recency_half_life_day': 30.0,
 'include_feed_header': True,
 'feed_header_keys': ['title', 'header', 'name', 'feed_title'],
 'params_path': 'parameters/parameters.yaml',
 'feed_index_dir': 'artifacts/feed_index',
 'user_bundle_dir': 'artifacts/user_query_bundles',
 'score_weights_path': 'parameters/retrieval_score_weights.yaml',
 'score_aggregation': {'enabled': True,
  'mode': 'linear',
  'weights': {'vector_score': 0.6,
   'language_match': 0.3,
   'recency': 0.05,
   'popularity': 0.05},
  'clamp_inputs': True,
  'renormalize': True,
  'missing_subscore_value': 0.0,
  'tie_breakers': ['vector_score', 'recency'],
  'applied': True},
 'exclude_seen_enabled': True,
 'exclude_seen_count': 0,
 'exclude_seen_interactions_path': 'data/interactions.csv',
 'exclude_seen_ev

In [640]:
res = OnlineRetrievalResult(student_id=student_id, candidates=candidates, meta=meta)

In [637]:
preview_n = 5

In [644]:
def _format_candidate_line(c: Dict[str, Any], ranking_key: str) -> str:
    """
    Human-readable one-liner for CLI printing.

    Prefer ranking_key (vector_score or final_score) if present, with fallback.
    """
    fid = c.get("feed_id")
    header = c.get("feed_header")

    score_val = c.get(ranking_key, c.get("vector_score"))
    try:
        sf = float(score_val)
    except Exception:
        sf = 0.0

    if header:
        return f"- {fid} | {header} | score={sf:.4f}"
    return f"- {fid} | score={sf:.4f}"

In [645]:
preview_n = int(max(0, preview_n))
preview = res.candidates[:preview_n]

ranking_key = str(res.meta.get("ranking_score_key") or "vector_score")

# Optional: print human-readable list first
if print_header_final:
    print(f"=== Preview (top {len(preview)}) ===")
    for c in preview:
        print(_format_candidate_line(c, ranking_key))

# Always print JSON summary (stable for tooling)
print(
    json.dumps(
        {
            "student_id": res.student_id,
            "num_candidates": len(res.candidates),
            "meta": res.meta,
            "candidates_preview": preview,
        },
        ensure_ascii=False,
        indent=2,
    )
)

=== Preview (top 5) ===
- TH_F001 | แนวทางทำพอร์ต Data Analyst ด้วยโปรเจกต์ Python และ SQL | score=0.5952
- TH_F003 | เทคนิคเตรียมสัมภาษณ์ฝึกงาน Data Analyst (SQL + Python) | score=0.5820
- TH_F023 | Data Analyst Intern: ตัวอย่างโจทย์ Take-home และวิธีทำให้ดูโปร | score=0.5483
- TH_F008 | รวมโจทย์ SQL ฝึกสัมภาษณ์ระดับฝึกงาน (พร้อมเฉลยแนวคิด) | score=0.5455
- TH_F006 | พื้นฐานสถิติที่ใช้บ่อยในงาน Data Analyst (พร้อมตัวอย่าง) | score=0.5439
{
  "student_id": "stu_p001",
  "num_candidates": 40,
  "meta": {
    "top_k_per_query": 30,
    "max_candidates": 200,
    "num_queries": 5,
    "dim": 768,
    "bundle_path": "stu_p001.json",
    "agg_mode": "WEIGHTED_MAX",
    "missing_npy_policy": "HARD_FAIL",
    "index_cached": true,
    "user_lang": "th",
    "recency_half_life_day": 30.0,
    "include_feed_header": true,
    "feed_header_keys": [
      "title",
      "header",
      "name",
      "feed_title"
    ],
    "params_path": "parameters/parameters.yaml",
    "feed_index_dir": "artifa